# STIR-Net V1 — 09 spatial corruption ablation

This is the final source-localization experiment before changing STIR-Net V1.

We already established that the current physical-aware CNN can learn the same BlastoSPIM scene in isolation. This notebook asks:

> **Does temporal/co-reasoning itself destroy the useful spatial representation, or does the full query/native-mask objective drive the shared spatial weights into a bad state?**

It evaluates four states on the exact same all-cell scene:

```text
A
Notebook 08 physical-aware spatial checkpoint
encoder -> decoder -> dense heads

B0
same good spatial weights + fresh temporal/CR modules
evaluated before any CR adaptation

B1
same model trained only with dense spatial losses through:
encoder -> temporal -> CR1 -> decoder -> CR2 -> decoder -> dense heads

C-active
real full STIR-Net checkpoint step 25 with CR active

C-bypass
the exact same step-25 weights, but CR1/CR2 bypassed
for the dense spatial evaluation
```

No query builder, query decoder, Hungarian matcher, native-mask renderer, or Gaussian prior is used in A/B dense-only training.

No source files are modified or monkey-patched.


In [ ]:
from pathlib import Path
import gc
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _repo_root,
    _reduced_config,
    build_real_batch,
)
from learned.stirnet.training.checkpoint import load_checkpoint

SEED = 40266

CR_DENSE_ONLY_STEPS = 20
EVAL_EVERY = 5

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0

AMP_DTYPE = torch.float16
GRAD_SCALER_INITIAL_SCALE = 1024.0
METRIC_CHUNK_VOXELS = 524_288

REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

SPATIAL_BASELINE_PATH = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "spatial_backbone_isolation"
    / "08_blastospim_first_overfit"
    / "physical_aware_final.pt"
)

FULL_CHECKPOINT_PATH = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "05_same_sample"
    / "checkpoint_step_025.pt"
)

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "spatial_corruption_ablation"
    / "09_blastospim_first_overfit"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 09 requires CUDA.")

device = torch.device("cuda")

print("Repository              :", REPO_ROOT)
print("Data                    :", DATA_DIR)
print("Notebook 08 baseline    :", SPATIAL_BASELINE_PATH)
print("Full step-25 checkpoint :", FULL_CHECKPOINT_PATH)
print("Output                  :", RUN_DIR)
print("GPU                     :", torch.cuda.get_device_name(0))

assert DATA_DIR.exists(), DATA_DIR
assert SPATIAL_BASELINE_PATH.exists(), (
    "Run Notebook 08 first; its physical-aware checkpoint is required:\n"
    f"{SPATIAL_BASELINE_PATH}"
)
assert FULL_CHECKPOINT_PATH.exists(), FULL_CHECKPOINT_PATH


## 1. Rebuild the exact previous all-cell BlastoSPIM sample

In [ ]:
batch, sample = build_real_batch(DATA_DIR)
targets = batch["targets"]
cfg = _reduced_config()

print("ROI shape          :", sample["roi_shape"])
print("Current cells      :", sample["current_count"])
print("GT cells           :", sample["target_count"])
print("Graph nodes        :", sample["graph_nodes"])
print("Temporal tracklets :", sample["temporal_tracklets"])
print("Required queries   :", sample["required_queries"])

assert sample["current_count"] == 36
assert sample["target_count"] == 33
assert sample["temporal_tracklets"] == 52
assert sample["required_queries"] == 132


## 2. Device helper

GT targets remain CPU-backed. Only model inputs are moved to CUDA.


In [ ]:
def make_device_batch():
    result = {}

    for key, value in batch.items():
        if key == "targets":
            result[key] = value
        elif torch.is_tensor(value):
            if key == "spatial_inputs":
                result[key] = value.to(
                    device=device,
                    dtype=AMP_DTYPE,
                    non_blocking=True,
                )
            elif key == "instance_labels":
                result[key] = value.to(
                    device=device,
                    dtype=torch.int32,
                    non_blocking=True,
                )
            else:
                result[key] = value.to(
                    device=device,
                    non_blocking=True,
                )
        else:
            result[key] = value

    return result


## 3. Two dense forward paths through the current model

`forward_dense_bypass_cr` uses the same spatial encoder/decoder/dense heads while skipping CR1/CR2.

`forward_dense_with_cr` reproduces the actual spatial + temporal + CR1 + CR2 path, but stops before the query system.


In [ ]:
def forward_dense_bypass_cr(model, b):
    acq = model.acquisition(
        b["spacing_um"],
        b["dref_um"],
    )

    pyramid = model.encoder(
        b["spatial_inputs"],
        b["spacing_um"],
        acq,
        b.get("spatial_padding_mask"),
    )

    e3 = pyramid.features[3]

    e2 = model.decoder.decode_to_e2(
        e3,
        pyramid,
        acq,
    )

    d1, d0, _ = model.decoder.decode_from_e2(
        e2,
        pyramid,
        acq,
    )

    dense = model.dense_heads(d0)

    return {
        "dense": dense,
        "pyramid": pyramid,
        "e3": e3,
        "e2": e2,
        "d1": d1,
        "d0": d0,
        "temporal": None,
    }


def forward_dense_with_cr(model, b):
    acq = model.acquisition(
        b["spacing_um"],
        b["dref_um"],
    )

    pyramid = model.encoder(
        b["spatial_inputs"],
        b["spacing_um"],
        acq,
        b.get("spatial_padding_mask"),
    )

    temporal = model._build_temporal(
        b["graph_x"],
        b["graph_edge_index"],
        b["graph_edge_attr"],
        b["tracklet_id"],
        b["temporal_ref_um"],
        b["temporal_status"],
        b["hypothesis_edge_index"],
        b["hypothesis_edge_attr"],
        b["temporal_batch"],
        b["dref_um"],
    )

    e3, temporal = model.cr1(
        pyramid.features[3],
        pyramid.spacings_um[3],
        temporal,
        b["dref_um"],
        acq,
        (
            pyramid.padding_masks[3]
            if pyramid.padding_masks
            else None
        ),
    )

    e2 = model.decoder.decode_to_e2(
        e3,
        pyramid,
        acq,
    )

    e2, temporal = model.cr2(
        e2,
        pyramid.spacings_um[2],
        temporal,
        b["dref_um"],
        acq,
        (
            pyramid.padding_masks[2]
            if pyramid.padding_masks
            else None
        ),
    )

    d1, d0, _ = model.decoder.decode_from_e2(
        e2,
        pyramid,
        acq,
    )

    dense = model.dense_heads(d0)

    return {
        "dense": dense,
        "pyramid": pyramid,
        "e3": e3,
        "e2": e2,
        "d1": d1,
        "d0": d0,
        "temporal": temporal,
    }


## 4. Use the exact current dense losses

In [ ]:
criterion = RefinementCriterion(
    cfg.losses,
    cfg.queries,
    cfg.training,
).to(device)


def dense_loss(dense_outputs):
    foreground, center_heatmap, boundary = criterion._dense_losses(
        dense_outputs,
        targets,
    )

    total = (
        cfg.losses.foreground * foreground
        + cfg.losses.center_heatmap * center_heatmap
        + cfg.losses.boundary * boundary
    )

    return {
        "loss": total,
        "foreground": foreground,
        "center_heatmap": center_heatmap,
        "boundary": boundary,
    }


print("Dense loss weights:")
print(" foreground     =", cfg.losses.foreground)
print(" center_heatmap =", cfg.losses.center_heatmap)
print(" boundary       =", cfg.losses.boundary)


## 5. Bounded evaluation metrics

In [ ]:
@torch.no_grad()
def binary_metrics(
    logits,
    target_cpu,
    *,
    threshold=0.5,
):
    flat_logits = logits.detach().float().reshape(-1)
    flat_target = torch.as_tensor(target_cpu).reshape(-1)

    intersection = 0
    predicted_count = 0
    target_count = 0

    positive_sum = 0.0
    positive_count = 0

    negative_sum = 0.0
    negative_count = 0

    for start in range(
        0,
        flat_logits.numel(),
        METRIC_CHUNK_VOXELS,
    ):
        end = min(
            start + METRIC_CHUNK_VOXELS,
            flat_logits.numel(),
        )

        probability = torch.sigmoid(
            flat_logits[start:end]
        )

        target = flat_target[start:end].to(
            device=logits.device,
            dtype=torch.bool,
            non_blocking=True,
        )

        predicted = probability > threshold

        intersection += int(
            (predicted & target).sum().cpu()
        )
        predicted_count += int(
            predicted.sum().cpu()
        )
        target_count += int(
            target.sum().cpu()
        )

        if target.any():
            positive_sum += float(
                probability[target].sum().cpu()
            )
            positive_count += int(
                target.sum().cpu()
            )

        negative = ~target
        if negative.any():
            negative_sum += float(
                probability[negative].sum().cpu()
            )
            negative_count += int(
                negative.sum().cpu()
            )

    return {
        "dice": (
            2.0 * intersection
            / max(
                predicted_count + target_count,
                1,
            )
        ),
        "mean_prob_positive": (
            positive_sum
            / max(positive_count, 1)
        ),
        "mean_prob_negative": (
            negative_sum
            / max(negative_count, 1)
        ),
    }


@torch.no_grad()
def center_metrics(logits, target_cpu):
    flat_logits = logits.detach().float().reshape(-1)
    flat_target = torch.as_tensor(target_cpu).reshape(-1)

    squared_error = 0.0
    total = 0

    high_sum = 0.0
    high_count = 0

    low_sum = 0.0
    low_count = 0

    for start in range(
        0,
        flat_logits.numel(),
        METRIC_CHUNK_VOXELS,
    ):
        end = min(
            start + METRIC_CHUNK_VOXELS,
            flat_logits.numel(),
        )

        probability = torch.sigmoid(
            flat_logits[start:end]
        )

        target = flat_target[start:end].to(
            device=logits.device,
            dtype=torch.float32,
            non_blocking=True,
        )

        squared_error += float(
            (probability - target)
            .square()
            .sum()
            .cpu()
        )
        total += end - start

        high = target > 0.5
        low = target < 0.05

        if high.any():
            high_sum += float(
                probability[high].sum().cpu()
            )
            high_count += int(
                high.sum().cpu()
            )

        if low.any():
            low_sum += float(
                probability[low].sum().cpu()
            )
            low_count += int(
                low.sum().cpu()
            )

    return {
        "mse": squared_error / max(total, 1),
        "mean_prob_high_target": (
            high_sum / max(high_count, 1)
        ),
        "mean_prob_low_target": (
            low_sum / max(low_count, 1)
        ),
    }


@torch.no_grad()
def evaluate_dense(dense):
    foreground = binary_metrics(
        dense["foreground_logits"][0, 0],
        targets[0]["foreground"],
    )

    boundary = binary_metrics(
        dense["boundary_logits"][0, 0],
        (
            torch.as_tensor(
                targets[0]["boundary"]
            ) > 0.5
        ),
    )

    center = center_metrics(
        dense["center_heatmap_logits"][0, 0],
        targets[0]["center_heatmap"],
    )

    return {
        "foreground_dice": float(
            foreground["dice"]
        ),
        "foreground_prob_inside": float(
            foreground["mean_prob_positive"]
        ),
        "foreground_prob_outside": float(
            foreground["mean_prob_negative"]
        ),
        "boundary_dice": float(
            boundary["dice"]
        ),
        "boundary_prob_on": float(
            boundary["mean_prob_positive"]
        ),
        "boundary_prob_off": float(
            boundary["mean_prob_negative"]
        ),
        "center_mse": float(
            center["mse"]
        ),
        "center_prob_high_target": float(
            center["mean_prob_high_target"]
        ),
        "center_prob_low_target": float(
            center["mean_prob_low_target"]
        ),
    }


@torch.no_grad()
def evaluate_model(model, forward_fn):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.eval()
    criterion.eval()

    b = make_device_batch()

    started = time.perf_counter()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        result = forward_fn(
            model,
            b,
        )
        losses = dense_loss(
            result["dense"]
        )

    torch.cuda.synchronize()

    metrics = evaluate_dense(
        result["dense"]
    )

    row = {
        key: float(
            value.detach().float().cpu()
        )
        for key, value in losses.items()
    }
    row.update(metrics)

    row["seconds"] = float(
        time.perf_counter() - started
    )
    row["peak_cuda_gib"] = float(
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    del result, b
    gc.collect()
    torch.cuda.empty_cache()

    return row


## 6. Helper for loading Notebook 08 spatial weights into a full `StirNet`

Notebook 08 saved only the useful spatial experiment state. Missing temporal/query modules are expected.


In [ ]:
def extract_model_state(payload):
    if (
        isinstance(payload, dict)
        and "model" in payload
        and isinstance(payload["model"], dict)
    ):
        return payload["model"]
    return payload


def load_spatial_baseline(model):
    payload = torch.load(
        SPATIAL_BASELINE_PATH,
        map_location="cpu",
        weights_only=False,
    )

    state = extract_model_state(payload)

    result = model.load_state_dict(
        state,
        strict=False,
    )

    if result.unexpected_keys:
        raise RuntimeError(
            "Unexpected Notebook 08 keys: "
            + ", ".join(
                result.unexpected_keys
            )
        )

    print(
        "Notebook 08 weights loaded.",
        "Missing full-model keys:",
        len(result.missing_keys),
    )

    return payload


## 7. A — verify the good spatial-only checkpoint

In [ ]:
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

model_a = StirNet(cfg)
load_spatial_baseline(model_a)
model_a = model_a.to(device)

a_metrics = evaluate_model(
    model_a,
    forward_dense_bypass_cr,
)

display(
    pd.DataFrame(
        [
            {
                "state": "A_spatial_only",
                **a_metrics,
            }
        ]
    )
)

model_a.cpu()
del model_a
gc.collect()
torch.cuda.empty_cache()


## 8. B0 — insert fresh temporal + co-reasoning modules into the good spatial model

This measures the immediate perturbation before any CR adaptation.


In [ ]:
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

model_b = StirNet(cfg)
load_spatial_baseline(model_b)
model_b = model_b.to(device)

b0_active = evaluate_model(
    model_b,
    forward_dense_with_cr,
)

b0_bypass = evaluate_model(
    model_b,
    forward_dense_bypass_cr,
)

display(
    pd.DataFrame(
        [
            {
                "state": "B0_active_CR",
                **b0_active,
            },
            {
                "state": "B0_bypass_CR",
                **b0_bypass,
            },
        ]
    )
)


## 9. B1 — dense-only training through temporal + CR1 + CR2

Only the three dense spatial losses are used. The query system and native mask path are absent.


In [ ]:
optimizer_b = torch.optim.AdamW(
    model_b.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scaler_b = torch.amp.GradScaler(
    "cuda",
    init_scale=GRAD_SCALER_INITIAL_SCALE,
)

b_history = []

initial_eval = evaluate_model(
    model_b,
    forward_dense_with_cr,
)

b_history.append(
    {
        "step": 0,
        **initial_eval,
    }
)

print(
    "step 000 | "
    f"loss={initial_eval['loss']:.6f} | "
    f"fg={initial_eval['foreground_dice']:.4f} | "
    f"boundary={initial_eval['boundary_dice']:.4f}"
)

for step in range(
    1,
    CR_DENSE_ONLY_STEPS + 1,
):
    model_b.train()
    criterion.train()

    optimizer_b.zero_grad(
        set_to_none=True
    )

    b = make_device_batch()

    started = time.perf_counter()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        result = forward_dense_with_cr(
            model_b,
            b,
        )

        losses = dense_loss(
            result["dense"]
        )

        loss = losses["loss"]

    scaler_b.scale(loss).backward()
    scaler_b.unscale_(optimizer_b)

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model_b.parameters(),
        MAX_GRAD_NORM,
    )

    scaler_b.step(optimizer_b)
    scaler_b.update()

    torch.cuda.synchronize()

    if (
        step % EVAL_EVERY == 0
        or step == CR_DENSE_ONLY_STEPS
    ):
        evaluation = evaluate_model(
            model_b,
            forward_dense_with_cr,
        )

        evaluation.update(
            {
                "step": step,
                "grad_norm": float(
                    torch.as_tensor(
                        grad_norm
                    )
                    .detach()
                    .float()
                    .cpu()
                ),
                "scale": float(
                    scaler_b.get_scale()
                ),
                "train_step_seconds": float(
                    time.perf_counter() - started
                ),
            }
        )

        b_history.append(
            evaluation
        )

        print(
            f"step {step:03d} | "
            f"loss={evaluation['loss']:.6f} | "
            f"fg={evaluation['foreground_dice']:.4f} | "
            f"boundary={evaluation['boundary_dice']:.4f} | "
            f"fg in/out="
            f"{evaluation['foreground_prob_inside']:.3f}/"
            f"{evaluation['foreground_prob_outside']:.3f}"
        )

    del result, losses, loss, b

b_history_df = pd.DataFrame(
    b_history
)

display(b_history_df)


## 10. Evaluate B1 with the exact same trained weights, CR active vs bypassed


In [ ]:
b1_active = evaluate_model(
    model_b,
    forward_dense_with_cr,
)

b1_bypass = evaluate_model(
    model_b,
    forward_dense_bypass_cr,
)

display(
    pd.DataFrame(
        [
            {
                "state": "B1_active_CR",
                **b1_active,
            },
            {
                "state": "B1_bypass_CR",
                **b1_bypass,
            },
        ]
    )
)


## 11. Save B1 and free its GPU state

In [ ]:
B1_PATH = (
    RUN_DIR
    / "dense_only_coreasoning_final.pt"
)

torch.save(
    {
        "model": {
            key: value.detach().cpu()
            for key, value in model_b.state_dict().items()
        },
        "step": CR_DENSE_ONLY_STEPS,
        "config": cfg.to_dict(),
    },
    B1_PATH,
)

b_history_df.to_csv(
    RUN_DIR / "dense_only_coreasoning_history.csv",
    index=False,
)

print("Saved:", B1_PATH)

model_b.cpu()
del model_b, optimizer_b, scaler_b

gc.collect()
torch.cuda.empty_cache()


## 12. C — real full STIR-Net checkpoint step 25

Evaluate the same trained weights twice:

```text
C-active  = normal temporal + CR1 + CR2 dense path
C-bypass  = same encoder/decoder/dense weights, CR1/CR2 skipped
```

This is the key corruption-localization comparison.


In [ ]:
model_c = StirNet(cfg).to(device)

checkpoint_c = load_checkpoint(
    FULL_CHECKPOINT_PATH,
    model_c,
    map_location="cpu",
    strict=True,
)

print(
    "Loaded full checkpoint step:",
    checkpoint_c.get("step"),
)

c_active = evaluate_model(
    model_c,
    forward_dense_with_cr,
)

c_bypass = evaluate_model(
    model_c,
    forward_dense_bypass_cr,
)

display(
    pd.DataFrame(
        [
            {
                "state": "C_active_CR",
                **c_active,
            },
            {
                "state": "C_bypass_CR",
                **c_bypass,
            },
        ]
    )
)


## 13. Final comparison

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "state": "A_spatial_only_good",
            **a_metrics,
        },
        {
            "state": "B0_good_spatial_plus_fresh_CR",
            **b0_active,
        },
        {
            "state": "B1_dense_only_CR_trained",
            **b1_active,
        },
        {
            "state": "B1_same_weights_CR_bypassed",
            **b1_bypass,
        },
        {
            "state": "C_full_step25_CR_active",
            **c_active,
        },
        {
            "state": "C_full_step25_CR_bypassed",
            **c_bypass,
        },
    ]
)

display(
    comparison[
        [
            "state",
            "loss",
            "foreground_dice",
            "foreground_prob_inside",
            "foreground_prob_outside",
            "boundary_dice",
            "boundary_prob_on",
            "boundary_prob_off",
            "center_mse",
            "center_prob_high_target",
            "center_prob_low_target",
            "peak_cuda_gib",
        ]
    ]
)

comparison.to_csv(
    RUN_DIR / "comparison.csv",
    index=False,
)


## 14. Diagnostic ratios and interpretation

The thresholds below are only convenient heuristics. The table above is the actual evidence.


In [ ]:
def safe_ratio(a, b):
    return float(a) / max(
        float(b),
        1e-8,
    )


ratios = {
    "B0_fg_vs_A": safe_ratio(
        b0_active["foreground_dice"],
        a_metrics["foreground_dice"],
    ),
    "B1_fg_vs_A": safe_ratio(
        b1_active["foreground_dice"],
        a_metrics["foreground_dice"],
    ),
    "B1_boundary_vs_A": safe_ratio(
        b1_active["boundary_dice"],
        a_metrics["boundary_dice"],
    ),
    "C_bypass_fg_vs_active": safe_ratio(
        c_bypass["foreground_dice"],
        c_active["foreground_dice"],
    ),
    "C_bypass_boundary_vs_active": safe_ratio(
        c_bypass["boundary_dice"],
        c_active["boundary_dice"],
    ),
    "C_bypass_fg_vs_A": safe_ratio(
        c_bypass["foreground_dice"],
        a_metrics["foreground_dice"],
    ),
}

display(
    pd.DataFrame(
        [
            {
                "metric": key,
                "value": value,
            }
            for key, value in ratios.items()
        ]
    )
)

a_good = (
    a_metrics["foreground_dice"]
    >= 0.40
)

b1_recovers = (
    b1_active["foreground_dice"]
    >= 0.80
    * a_metrics["foreground_dice"]
)

c_bypass_big_gain = (
    c_bypass["foreground_dice"]
    >= c_active["foreground_dice"]
    + 0.10
)

print("\nAutomated reading:")

if a_good and b1_recovers:
    print(
        "1. Temporal + CR can preserve/learn useful spatial geometry "
        "when trained with dense losses only."
    )
else:
    print(
        "1. Temporal + CR did not recover the spatial baseline; "
        "co-reasoning itself remains a primary suspect."
    )

if c_bypass_big_gain:
    print(
        "2. Bypassing CR substantially restores the full checkpoint. "
        "The trained CR path is an active corruption point."
    )
else:
    print(
        "2. Bypassing CR does not substantially restore the full checkpoint. "
        "The shared spatial encoder/decoder/dense-head weights themselves "
        "were likely driven into a poor state by the full training objective."
    )


## 15. B1 dense-only co-reasoning learning curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    b_history_df["step"],
    b_history_df["loss"],
    marker="o",
)
plt.xlabel("Optimizer step")
plt.ylabel("Dense-only loss")
plt.title("Temporal + CR dense-only adaptation")
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    b_history_df["step"],
    b_history_df["foreground_dice"],
    marker="o",
    label="foreground Dice",
)
plt.plot(
    b_history_df["step"],
    b_history_df["boundary_dice"],
    marker="o",
    label="boundary Dice",
)
plt.xlabel("Optimizer step")
plt.ylabel("Dice")
plt.title("Spatial geometry through co-reasoning")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 16. Optional final 3D foreground sanity check

The following block reconstructs four foreground probability volumes:

```text
A spatial-only
B1 dense-only CR trained
C full checkpoint CR active
C full checkpoint CR bypassed
```


In [ ]:
@torch.no_grad()
def render_foreground(model, forward_fn):
    model.eval()

    b = make_device_batch()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        result = forward_fn(
            model,
            b,
        )

    foreground = (
        torch.sigmoid(
            result["dense"]["foreground_logits"][0, 0]
            .float()
        )
        .cpu()
        .numpy()
        .astype(np.float16)
    )

    del result, b
    gc.collect()
    torch.cuda.empty_cache()

    return foreground


# A
model_a_vis = StirNet(cfg)
load_spatial_baseline(model_a_vis)
model_a_vis = model_a_vis.to(device)

a_foreground = render_foreground(
    model_a_vis,
    forward_dense_bypass_cr,
)

model_a_vis.cpu()
del model_a_vis
gc.collect()
torch.cuda.empty_cache()


# B1
payload_b1 = torch.load(
    B1_PATH,
    map_location="cpu",
    weights_only=False,
)

model_b_vis = StirNet(cfg).to(device)
model_b_vis.load_state_dict(
    payload_b1["model"],
    strict=True,
)

b1_foreground = render_foreground(
    model_b_vis,
    forward_dense_with_cr,
)

model_b_vis.cpu()
del model_b_vis
gc.collect()
torch.cuda.empty_cache()


# C
c_active_foreground = render_foreground(
    model_c,
    forward_dense_with_cr,
)

c_bypass_foreground = render_foreground(
    model_c,
    forward_dense_bypass_cr,
)

print("Foreground volumes ready.")


In [ ]:
raw = (
    batch["spatial_inputs"][0, 0]
    .float()
    .cpu()
    .numpy()
)

gt_labels = (
    torch.as_tensor(
        targets[0]["label_map"]
    )
    .cpu()
    .numpy()
)

z = raw.shape[0] // 2

fig, axes = plt.subplots(
    2,
    3,
    figsize=(16, 10),
)

axes[0, 0].imshow(
    raw[z],
    cmap="gray",
)
axes[0, 0].set_title(f"Raw z={z}")

axes[0, 1].imshow(
    gt_labels[z] > 0,
)
axes[0, 1].set_title("GT foreground")

axes[0, 2].imshow(
    a_foreground[z],
    vmin=0,
    vmax=1,
)
axes[0, 2].set_title("A spatial-only")

axes[1, 0].imshow(
    b1_foreground[z],
    vmin=0,
    vmax=1,
)
axes[1, 0].set_title("B1 dense-only CR")

axes[1, 1].imshow(
    c_active_foreground[z],
    vmin=0,
    vmax=1,
)
axes[1, 1].set_title("C step25 CR active")

axes[1, 2].imshow(
    c_bypass_foreground[z],
    vmin=0,
    vmax=1,
)
axes[1, 2].set_title("C step25 CR bypass")

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 17. Save final summary and clear CUDA

In [ ]:
summary = {
    "sample": sample,
    "cr_dense_only_steps": CR_DENSE_ONLY_STEPS,
    "A_spatial_only": a_metrics,
    "B0_active_before_adaptation": b0_active,
    "B0_bypass_before_adaptation": b0_bypass,
    "B1_active_after_dense_only_training": b1_active,
    "B1_bypass_after_dense_only_training": b1_bypass,
    "C_full_step25_active": c_active,
    "C_full_step25_bypass": c_bypass,
    "ratios": ratios,
}

with (
    RUN_DIR / "summary.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
        default=lambda value: (
            value.tolist()
            if isinstance(value, np.ndarray)
            else value
        ),
    )

print("Saved:")
print(RUN_DIR / "summary.json")
print(RUN_DIR / "comparison.csv")
print(RUN_DIR / "dense_only_coreasoning_history.csv")
print(B1_PATH)

model_c.cpu()
del model_c

gc.collect()
torch.cuda.empty_cache()

print("CUDA cache cleared.")
